In [4]:
import ee

ee.Authenticate()
ee.Initialize(project="nigeria-flood-prediction")

countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
nigeria = countries.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))

rainfall = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
rainfall_filtered = rainfall.filterDate("2016-01-01", "2025-12-31")
rainfall_total = rainfall_filtered.sum()
rainfall_nigeria = rainfall_total.clip(nigeria)

In [5]:
import geemap
Map = geemap.Map()
vis_params = {"min":1220, "max": 37920, "palette": ["white", "blue"]}
Map.addLayer(rainfall_nigeria, vis_params, "rainfall")
Map.centerObject(nigeria, 6)
Map

Map(center=[9.589444610453373, 8.089338153274143], controls=(WidgetControl(options=['position', 'transparent_b…

In [7]:
stats = rainfall_nigeria.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=nigeria,
    scale=5000,
    maxPixels=1e9
)
print(stats.getInfo())

{'precipitation_max': 37920.06541033089, 'precipitation_min': 1220.5925972329878}


In [8]:
elevation = ee.Image("USGS/SRTMGL1_003")
elevation_nigeria = elevation.clip(nigeria)

stats = elevation_nigeria.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=nigeria,
    scale=5000,
    maxPixels=1e9
)
print(stats.getInfo())

{'elevation_max': 1787, 'elevation_min': -2}


In [9]:
elevation_vis_params = {"min": 0, "max": 1800, "palette": ["yellow", "brown"]}
Map.addLayer(elevation_nigeria, elevation_vis_params, "elevation")
Map.centerObject(nigeria, 6)
Map

Map(bottom=8054.0, center=[9.589444610453368, 8.089338153274143], controls=(WidgetControl(options=['position',…

In [10]:
landcover = ee.Image("ESA/WorldCover/v100/2020")
landcover_nigeria = landcover.clip(nigeria)

stats = landcover_nigeria.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=nigeria,
    scale=5000,
    maxPixels=1e9
)
print(stats.getInfo())

{'Map_max': 95, 'Map_min': 10}


In [11]:
landcover_vis_params = {
    "min": 10,
    "max": 95,
    "palette": [
        "006400",  # 10 tree cover - dark green
        "ffbb22",  # 20 shrubland - orange
        "ffff4c",  # 30 grassland - yellow
        "f096ff",  # 40 cropland - pink
        "fa0000",  # 50 built-up - red
        "b4b4b4",  # 60 bare/sparse - grey
        "f0f0f0",  # 70 snow/ice - white
        "0064c8",  # 80 water - blue
        "0096a0",  # 90 wetland - teal
        "00cf75",  # 95 mangroves - light green
    ]
}
Map.addLayer(landcover_nigeria, landcover_vis_params, "landcover")
Map.centerObject(nigeria, 6)
Map

Map(bottom=8054.0, center=[9.589444610453373, 8.089338153274143], controls=(WidgetControl(options=['position',…

In [12]:
surface_water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
surface_water_nigeria = surface_water.clip(nigeria)

stats = surface_water_nigeria.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=nigeria,
    scale=5000,
    maxPixels=1e9
)
print(stats.getInfo())

{'change_abs_max': 86, 'change_abs_min': -128, 'change_norm_max': 100, 'change_norm_min': -128, 'max_extent_max': 1, 'max_extent_min': 0, 'occurrence_max': 100, 'occurrence_min': 0, 'recurrence_max': 100, 'recurrence_min': 11, 'seasonality_max': 12, 'seasonality_min': 1, 'transition_max': 10, 'transition_min': 1}


In [13]:
water_occurrence = surface_water_nigeria.select("occurrence")
water_mask = water_occurrence.gt(50)
distance_to_water = water_mask.fastDistanceTransform().sqrt()
distance_to_water_nigeria = distance_to_water.clip(nigeria)
distance_to_water_meters = distance_to_water_nigeria.multiply(30)

In [16]:
stats = distance_to_water_meters.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=nigeria,
    scale=5000,
    maxPixels=1e9
)
print(stats.getInfo())

{'distance_max': 874.642784226795, 'distance_min': 0}


In [17]:
water_vis_params = {"min": 0, "max": 875, "palette": ["darkblue", "white"]}
Map.addLayer(distance_to_water_meters, water_vis_params, "distance to water")
Map.centerObject(nigeria, 6)
Map

Map(bottom=8054.0, center=[9.589444610453373, 8.089338153274143], controls=(WidgetControl(options=['position',…

In [18]:
combined = rainfall_nigeria.rename("rainfall") \
    .addBands(elevation_nigeria.rename("elevation")) \
    .addBands(landcover_nigeria.rename("landcover")) \
    .addBands(distance_to_water_meters.rename("distance_to_water"))

In [19]:
combined = combined.addBands(flood_label.rename("flooded"))

NameError: name 'flood_label' is not defined

In [20]:
sample_points = ee.FeatureCollection.randomPoints(region=nigeria, points=8000)

In [21]:
sampled_data = combined.sampleRegions(
    collection=sample_points,
    scale=5000,
    geometries=True
)

In [22]:
print(sampled_data.first().getInfo())

EEException: Computation timed out.

In [18]:
df_gee = geemap.ee_to_df(sampled_data, remove_geom=False)
print(df_gee.shape)


(7995, 5)


In [19]:
from pathlib import Path
print(Path.cwd())

c:\Users\User\Desktop\FLOOD-RISK-PREDICTOR\Notebooks


In [20]:
PROJECT_ROOT = Path.cwd().parent
print(PROJECT_ROOT)

c:\Users\User\Desktop\FLOOD-RISK-PREDICTOR


In [21]:
GEE_DATA_PATH = PROJECT_ROOT / "data" / "gee" / "flood_features_nigeria.csv"
df_gee.to_csv(GEE_DATA_PATH, index=False)

In [22]:
flood_events = ee.ImageCollection("GLOBAL_FLOOD_DB/MODIS_EVENTS/V1")
flood_events_nigeria = flood_events.filter(ee.Filter.bounds(nigeria))
flood_band = flood_events_nigeria.select("flooded")
flood_count = flood_band.sum()
flood_count_nigeria = flood_count.clip(nigeria)

In [23]:
stats = flood_count_nigeria.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=nigeria,
    scale=5000,
    maxPixels=1e9
)
print(stats.getInfo())

{'flooded_max': 11, 'flooded_min': 0}


In [24]:
flood_events = ee.ImageCollection("GLOBAL_FLOOD_DB/MODIS_EVENTS/V1")
flood_events_nigeria = flood_events.filter(ee.Filter.bounds(nigeria))
flood_band = flood_events_nigeria.select("flooded")
flood_count = flood_band.sum()
flood_count_nigeria = flood_count.clip(nigeria)
flood_label = flood_count_nigeria.gt(0)

In [25]:
flood_label = flood_count_nigeria.gt(0)
combined = combined.addBands(flood_label.rename("flooded"))

In [26]:
print(combined.bandNames().getInfo())

['rainfall', 'elevation', 'landcover', 'distance_to_water', 'flooded']


In [27]:
sampled_data = combined.sampleRegions(
    collection=sample_points,
    scale=5000,
    geometries=True
)

In [28]:
df_gee = geemap.ee_to_df(sampled_data, remove_geom=False)
print(df_gee.shape)
print(df_gee.head())
print(df_gee["flooded"].value_counts())


(7995, 6)
                                                 geo  distance_to_water  \
0  {'type': 'Point', 'coordinates': [8.1971269675...         161.554944   
1  {'type': 'Point', 'coordinates': [3.9301293680...         216.333077   
2  {'type': 'Point', 'coordinates': [5.4123495868...         108.166538   
3  {'type': 'Point', 'coordinates': [9.4098526011...         240.000000   
4  {'type': 'Point', 'coordinates': [10.667493998...         318.904374   

   elevation  flooded  landcover      rainfall  
0        730        0         20  13367.705167  
1        339        0         40  12014.300833  
2        276        0         40   6776.401724  
3        381        0         40   6869.194950  
4        354        0         40   5482.298329  
flooded
0    7717
1     278
Name: count, dtype: int64


In [29]:

from sklearn.model_selection import train_test_split
X = df_gee.drop(columns=["flooded","geo"] )
y = df_gee["flooded"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)
print(X_train.shape)
print(X_test.shape)

(5996, 4)
(1999, 4)


In [30]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [31]:
predictions = model.predict(X_test)
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[1925    4]
 [  68    2]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1929
           1       0.33      0.03      0.05        70

    accuracy                           0.96      1999
   macro avg       0.65      0.51      0.52      1999
weighted avg       0.94      0.96      0.95      1999



model = LogisticRegression(class_weight="balanced")
model.fit(X_train, y_train)

In [32]:
model = LogisticRegression(class_weight="balanced")
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[1540  389]
 [   8   62]]
              precision    recall  f1-score   support

           0       0.99      0.80      0.89      1929
           1       0.14      0.89      0.24        70

    accuracy                           0.80      1999
   macro avg       0.57      0.84      0.56      1999
weighted avg       0.96      0.80      0.86      1999



In [33]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(class_weight="balanced")
model.fit(X_train, y_train)

,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fa

In [34]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(class_weight="balanced")
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[1905   24]
 [  23   47]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1929
           1       0.66      0.67      0.67        70

    accuracy                           0.98      1999
   macro avg       0.83      0.83      0.83      1999
weighted avg       0.98      0.98      0.98      1999



In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

model = LogisticRegression(class_weight="balanced")
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[1540  389]
 [   8   62]]
              precision    recall  f1-score   support

           0       0.99      0.80      0.89      1929
           1       0.14      0.89      0.24        70

    accuracy                           0.80      1999
   macro avg       0.57      0.84      0.56      1999
weighted avg       0.96      0.80      0.86      1999



In [36]:
print(df_gee.groupby("flooded")["distance_to_water"].mean())

flooded
0    222.629922
1     64.278768
Name: distance_to_water, dtype: float64


In [40]:
print(df_gee.groupby("flooded")["landcover"].value_counts())

flooded  landcover
0        40           3281
         10           1613
         20           1442
         30           1169
         95             85
         50             61
         90             39
         80             23
         60              4
1        40            121
         80             43
         30             37
         10             33
         20             20
         90             16
         50              5
         95              2
         60              1
Name: count, dtype: int64


In [37]:
print(X_train.duplicated().sum())
print(X_test.duplicated().sum())

430
54


In [38]:
df_gee = df_gee.drop_duplicates(subset=["rainfall", "elevation", "landcover", "distance_to_water", "flooded"])
print(df_gee.shape)


(7221, 6)


In [39]:
X = df_gee.drop(columns=["flooded", "geo"])
y = df_gee["flooded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)

(5415, 4)
(1806, 4)


In [40]:
model = LogisticRegression(class_weight="balanced")
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[1417  325]
 [  15   49]]
              precision    recall  f1-score   support

           0       0.99      0.81      0.89      1742
           1       0.13      0.77      0.22        64

    accuracy                           0.81      1806
   macro avg       0.56      0.79      0.56      1806
weighted avg       0.96      0.81      0.87      1806



In [41]:
print(df_gee.groupby("flooded")["distance_to_water"].describe())


          count        mean         std  min       25%         50%  \
flooded                                                              
0        6966.0  222.702632  163.696067  0.0  94.86833  182.482876   
1         255.0   64.169432  116.832918  0.0   0.00000   30.000000   

                75%         max  
flooded                          
0        318.904374  917.823512  
1         67.082039  690.000000  


In [42]:
print(df_gee[df_gee["landcover"] == 80]["flooded"].value_counts())

flooded
1    40
0    22
Name: count, dtype: int64


In [43]:
df_gee = df_gee[df_gee["landcover"] != 80]
print(df_gee.shape)
print(df_gee["flooded"].value_counts())

(7159, 6)
flooded
0    6944
1     215
Name: count, dtype: int64


In [44]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
X = df_gee.drop(columns=["flooded", "geo"])
y = df_gee["flooded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

model = LogisticRegression(class_weight="balanced")
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[1418  318]
 [  12   42]]
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1736
           1       0.12      0.78      0.20        54

    accuracy                           0.82      1790
   macro avg       0.55      0.80      0.55      1790
weighted avg       0.97      0.82      0.87      1790



In [45]:
import joblib

GEE_MODEL_PATH = PROJECT_ROOT / "models" / "flood_classifier_nigeria.pkl"
joblib.dump(model, GEE_MODEL_PATH)

['c:\\Users\\User\\Desktop\\FLOOD-RISK-PREDICTOR\\models\\flood_classifier_nigeria.pkl']

In [46]:
df_gee.to_csv(GEE_DATA_PATH, index=False)

In [47]:
probabilities = model.predict_proba(X_test)
print(probabilities[:5])

[[0.99690821 0.00309179]
 [0.92982532 0.07017468]
 [0.97293338 0.02706662]
 [0.94895355 0.05104645]
 [0.87645177 0.12354823]]


In [48]:
flood_probabilities = probabilities[:, 1]
print(flood_probabilities[:5])

[0.00309179 0.07017468 0.02706662 0.05104645 0.12354823]


In [49]:
threshold = 0.2
new_predictions = (flood_probabilities > threshold).astype(int)

In [50]:
print(confusion_matrix(y_test, new_predictions))
print(classification_report(y_test, new_predictions))

[[1024  712]
 [   6   48]]
              precision    recall  f1-score   support

           0       0.99      0.59      0.74      1736
           1       0.06      0.89      0.12        54

    accuracy                           0.60      1790
   macro avg       0.53      0.74      0.43      1790
weighted avg       0.97      0.60      0.72      1790



In [51]:
for threshold in [0.3, 0.35, 0.4, 0.45]:
    new_predictions = (flood_probabilities > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.3 ---
[[1180  556]
 [   7   47]]
              precision    recall  f1-score   support

           0       0.99      0.68      0.81      1736
           1       0.08      0.87      0.14        54

    accuracy                           0.69      1790
   macro avg       0.54      0.78      0.48      1790
weighted avg       0.97      0.69      0.79      1790


--- Threshold: 0.35 ---
[[1244  492]
 [   9   45]]
              precision    recall  f1-score   support

           0       0.99      0.72      0.83      1736
           1       0.08      0.83      0.15        54

    accuracy                           0.72      1790
   macro avg       0.54      0.77      0.49      1790
weighted avg       0.97      0.72      0.81      1790


--- Threshold: 0.4 ---
[[1316  420]
 [  10   44]]
              precision    recall  f1-score   support

           0       0.99      0.76      0.86      1736
           1       0.09      0.81      0.17        54

    accuracy                 

In [52]:
threshold = 0.33
new_predictions = (flood_probabilities > threshold).astype(int)

print(confusion_matrix(y_test, new_predictions))
print(classification_report(y_test, new_predictions))

[[1226  510]
 [   8   46]]
              precision    recall  f1-score   support

           0       0.99      0.71      0.83      1736
           1       0.08      0.85      0.15        54

    accuracy                           0.71      1790
   macro avg       0.54      0.78      0.49      1790
weighted avg       0.97      0.71      0.81      1790



In [54]:
threshold = 0.34
new_predictions = (flood_probabilities > threshold).astype(int)

print(confusion_matrix(y_test, new_predictions))
print(classification_report(y_test, new_predictions))

[[1239  497]
 [   8   46]]
              precision    recall  f1-score   support

           0       0.99      0.71      0.83      1736
           1       0.08      0.85      0.15        54

    accuracy                           0.72      1790
   macro avg       0.54      0.78      0.49      1790
weighted avg       0.97      0.72      0.81      1790



In [55]:
FLOOD_THRESHOLD = 0.34

final_predictions = (model.predict_proba(X_test)[:, 1] > FLOOD_THRESHOLD).astype(int)

print(confusion_matrix(y_test, final_predictions))
print(classification_report(y_test, final_predictions))

[[1239  497]
 [   8   46]]
              precision    recall  f1-score   support

           0       0.99      0.71      0.83      1736
           1       0.08      0.85      0.15        54

    accuracy                           0.72      1790
   macro avg       0.54      0.78      0.49      1790
weighted avg       0.97      0.72      0.81      1790



In [56]:
import joblib

GEE_MODEL_PATH = PROJECT_ROOT / "models" / "flood_classifier_nigeria.pkl"

model_package = {
    "model": model,
    "threshold": FLOOD_THRESHOLD
}

joblib.dump(model_package, GEE_MODEL_PATH)

['c:\\Users\\User\\Desktop\\FLOOD-RISK-PREDICTOR\\models\\flood_classifier_nigeria.pkl']

In [57]:
loaded_package = joblib.load(GEE_MODEL_PATH)
loaded_model = loaded_package["model"]
loaded_threshold = loaded_package["threshold"]

print(loaded_threshold)

0.34


In [58]:
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[1725   11]
 [  33   21]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1736
           1       0.66      0.39      0.49        54

    accuracy                           0.98      1790
   macro avg       0.82      0.69      0.74      1790
weighted avg       0.97      0.98      0.97      1790



In [59]:
flood_probabilities = model.predict_proba(X_test)[:, 1]
for threshold in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
    new_predictions = (flood_probabilities > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.05 ---
[[1651   85]
 [  18   36]]
              precision    recall  f1-score   support

           0       0.99      0.95      0.97      1736
           1       0.30      0.67      0.41        54

    accuracy                           0.94      1790
   macro avg       0.64      0.81      0.69      1790
weighted avg       0.97      0.94      0.95      1790


--- Threshold: 0.1 ---
[[1679   57]
 [  22   32]]
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1736
           1       0.36      0.59      0.45        54

    accuracy                           0.96      1790
   macro avg       0.67      0.78      0.71      1790
weighted avg       0.97      0.96      0.96      1790


--- Threshold: 0.15 ---
[[1701   35]
 [  24   30]]
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1736
           1       0.46      0.56      0.50        54

    accuracy                

In [60]:
for threshold in [0.01, 0.02, 0.03, 0.04]:
    new_predictions = (flood_probabilities > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.01 ---
[[1382  354]
 [   2   52]]
              precision    recall  f1-score   support

           0       1.00      0.80      0.89      1736
           1       0.13      0.96      0.23        54

    accuracy                           0.80      1790
   macro avg       0.56      0.88      0.56      1790
weighted avg       0.97      0.80      0.87      1790


--- Threshold: 0.02 ---
[[1577  159]
 [  10   44]]
              precision    recall  f1-score   support

           0       0.99      0.91      0.95      1736
           1       0.22      0.81      0.34        54

    accuracy                           0.91      1790
   macro avg       0.61      0.86      0.65      1790
weighted avg       0.97      0.91      0.93      1790


--- Threshold: 0.03 ---
[[1623  113]
 [  14   40]]
              precision    recall  f1-score   support

           0       0.99      0.93      0.96      1736
           1       0.26      0.74      0.39        54

    accuracy               

In [61]:
for threshold in [0.012, 0.014, 0.016, 0.018]:
    new_predictions = (flood_probabilities > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.012 ---
[[1430  306]
 [   2   52]]
              precision    recall  f1-score   support

           0       1.00      0.82      0.90      1736
           1       0.15      0.96      0.25        54

    accuracy                           0.83      1790
   macro avg       0.57      0.89      0.58      1790
weighted avg       0.97      0.83      0.88      1790


--- Threshold: 0.014 ---
[[1514  222]
 [   5   49]]
              precision    recall  f1-score   support

           0       1.00      0.87      0.93      1736
           1       0.18      0.91      0.30        54

    accuracy                           0.87      1790
   macro avg       0.59      0.89      0.62      1790
weighted avg       0.97      0.87      0.91      1790


--- Threshold: 0.016 ---
[[1547  189]
 [   8   46]]
              precision    recall  f1-score   support

           0       0.99      0.89      0.94      1736
           1       0.20      0.85      0.32        54

    accuracy            

In [25]:
forest = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")
loss_year = forest.select("lossyear")
forest_loss_recent = loss_year.gte(15)
forest_loss_nigeria = forest_loss_recent.clip(nigeria)

In [26]:
stats = forest_loss_nigeria.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=nigeria,
    scale=5000,
    maxPixels=1e9,
    bestEffort=True
)
print(stats.getInfo())

{'lossyear_max': 1, 'lossyear_min': 0}


In [27]:
forest_vis_params = {"min": 0, "max": 1, "palette": ["green", "red"]}
Map.addLayer(forest_loss_nigeria, forest_vis_params, "forest")
Map.centerObject(nigeria, 6)
Map

Map(bottom=8054.0, center=[9.589444610453373, 8.089338153274143], controls=(WidgetControl(options=['position',…

In [28]:
combined = combined.addBands(forest_loss_nigeria.rename("forest_loss"))
print(combined.bandNames().getInfo())

['rainfall', 'elevation', 'landcover', 'distance_to_water', 'forest_loss']


In [29]:
print(flood_label.getInfo())


NameError: name 'flood_label' is not defined

In [30]:
flood_events = ee.ImageCollection("GLOBAL_FLOOD_DB/MODIS_EVENTS/V1")
flood_events_nigeria = flood_events.filter(ee.Filter.bounds(nigeria))
flood_band = flood_events_nigeria.select("flooded")
flood_count = flood_band.sum()
flood_count_nigeria = flood_count.clip(nigeria)
flood_label = flood_count_nigeria.gt(0)

In [31]:
combined = combined.addBands(flood_label.rename("flooded"))
print(combined.bandNames().getInfo())

['rainfall', 'elevation', 'landcover', 'distance_to_water', 'forest_loss', 'flooded']


In [32]:
sampled_data = combined.sampleRegions(
    collection=sample_points,
    scale=5000,
    geometries=True
)

In [33]:
df_gee = geemap.ee_to_df(sampled_data, remove_geom=False)
print(df_gee.shape)
print(df_gee.head())

(6264, 7)
                                                 geo  distance_to_water  \
0  {'type': 'Point', 'coordinates': [8.1971269675...         161.554944   
1  {'type': 'Point', 'coordinates': [3.9301293680...         216.333077   
2  {'type': 'Point', 'coordinates': [5.2326865299...         127.279221   
3  {'type': 'Point', 'coordinates': [6.1759175783...         169.705627   
4  {'type': 'Point', 'coordinates': [5.1428550015...         624.259561   

   elevation  flooded  forest_loss  landcover      rainfall  
0        730        0            1         20  13367.705167  
1        339        0            0         40  12014.300833  
2         14        0            1         10  23738.312557  
3        197        0            1         20  11872.757912  
4        295        0            1         10  17677.749280  


In [34]:
df_gee = df_gee.drop_duplicates(subset=["rainfall", "elevation", "landcover", "distance_to_water", "forest_loss", "flooded"])
print(df_gee.shape)

(5658, 7)


In [35]:
df_gee = df_gee[df_gee["landcover"] != 80]
print(df_gee.shape)
print(df_gee["flooded"].value_counts())

(5608, 7)
flooded
0    5416
1     192
Name: count, dtype: int64


In [37]:
from sklearn.model_selection import train_test_split
X = df_gee.drop(columns=["flooded", "geo"])
y = df_gee["flooded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)

(4206, 5)
(1402, 5)


In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
model_lr = LogisticRegression(class_weight="balanced")
model_lr.fit(X_train, y_train)

flood_probabilities_lr = model_lr.predict_proba(X_test)[:, 1]
predictions_lr = (flood_probabilities_lr > 0.34).astype(int)

print("--- Logistic Regression (threshold 0.34) ---")
print(confusion_matrix(y_test, predictions_lr))
print(classification_report(y_test, predictions_lr))

--- Logistic Regression (threshold 0.34) ---
[[996 358]
 [  3  45]]
              precision    recall  f1-score   support

           0       1.00      0.74      0.85      1354
           1       0.11      0.94      0.20        48

    accuracy                           0.74      1402
   macro avg       0.55      0.84      0.52      1402
weighted avg       0.97      0.74      0.82      1402



In [43]:
from sklearn.ensemble import GradientBoostingClassifier
model_gb = GradientBoostingClassifier()
model_gb.fit(X_train, y_train)

flood_probabilities_gb = model_gb.predict_proba(X_test)[:, 1]
predictions_gb = (flood_probabilities_gb > 0.014).astype(int)

print("--- Gradient Boosting (threshold 0.014) ---")
print(confusion_matrix(y_test, predictions_gb))
print(classification_report(y_test, predictions_gb))

--- Gradient Boosting (threshold 0.014) ---
[[1210  144]
 [   3   45]]
              precision    recall  f1-score   support

           0       1.00      0.89      0.94      1354
           1       0.24      0.94      0.38        48

    accuracy                           0.90      1402
   macro avg       0.62      0.92      0.66      1402
weighted avg       0.97      0.90      0.92      1402



In [44]:
for threshold in [0.005, 0.008, 0.01, 0.012, 0.014, 0.016, 0.02]:
    new_predictions = (flood_probabilities_gb > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.005 ---
[[981 373]
 [  1  47]]
              precision    recall  f1-score   support

           0       1.00      0.72      0.84      1354
           1       0.11      0.98      0.20        48

    accuracy                           0.73      1402
   macro avg       0.56      0.85      0.52      1402
weighted avg       0.97      0.73      0.82      1402


--- Threshold: 0.008 ---
[[1135  219]
 [   2   46]]
              precision    recall  f1-score   support

           0       1.00      0.84      0.91      1354
           1       0.17      0.96      0.29        48

    accuracy                           0.84      1402
   macro avg       0.59      0.90      0.60      1402
weighted avg       0.97      0.84      0.89      1402


--- Threshold: 0.01 ---
[[1172  182]
 [   2   46]]
              precision    recall  f1-score   support

           0       1.00      0.87      0.93      1354
           1       0.20      0.96      0.33        48

    accuracy                 

In [45]:
for threshold in [0.01, 0.012, 0.016, 0.02]:
    new_predictions = (flood_probabilities_gb > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.01 ---
[[1172  182]
 [   2   46]]
              precision    recall  f1-score   support

           0       1.00      0.87      0.93      1354
           1       0.20      0.96      0.33        48

    accuracy                           0.87      1402
   macro avg       0.60      0.91      0.63      1402
weighted avg       0.97      0.87      0.91      1402


--- Threshold: 0.012 ---
[[1192  162]
 [   2   46]]
              precision    recall  f1-score   support

           0       1.00      0.88      0.94      1354
           1       0.22      0.96      0.36        48

    accuracy                           0.88      1402
   macro avg       0.61      0.92      0.65      1402
weighted avg       0.97      0.88      0.92      1402


--- Threshold: 0.016 ---
[[1216  138]
 [   3   45]]
              precision    recall  f1-score   support

           0       1.00      0.90      0.95      1354
           1       0.25      0.94      0.39        48

    accuracy             

In [46]:
for threshold in [0.016, 0.02]:
    new_predictions = (flood_probabilities_gb > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.016 ---
[[1216  138]
 [   3   45]]
              precision    recall  f1-score   support

           0       1.00      0.90      0.95      1354
           1       0.25      0.94      0.39        48

    accuracy                           0.90      1402
   macro avg       0.62      0.92      0.67      1402
weighted avg       0.97      0.90      0.93      1402


--- Threshold: 0.02 ---
[[1228  126]
 [   5   43]]
              precision    recall  f1-score   support

           0       1.00      0.91      0.95      1354
           1       0.25      0.90      0.40        48

    accuracy                           0.91      1402
   macro avg       0.63      0.90      0.67      1402
weighted avg       0.97      0.91      0.93      1402




In [49]:
slope = ee.Terrain.slope(elevation)
slope_nigeria = slope.clip(nigeria)

stats = slope_nigeria.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=nigeria,
    scale=5000,
    maxPixels=1e9,
    bestEffort=True
)
print(stats.getInfo())

{'slope_max': 8.472081519000987, 'slope_min': 0}


In [51]:
slope_vis_params = {"min": 0, "max": 8.5, "palette": ["purple", "green"]}
Map.addLayer(slope_nigeria, slope_vis_params, "slope")
Map.centerObject(nigeria, 6)
Map

Map(bottom=8054.0, center=[9.589444610453368, 8.089338153274143], controls=(WidgetControl(options=['position',…

In [52]:
print(combined.bandNames().getInfo())

['rainfall', 'elevation', 'landcover', 'distance_to_water', 'forest_loss', 'flooded']


In [53]:
combined = combined.addBands(slope_nigeria.rename("slope"))
print(combined.bandNames().getInfo())

['rainfall', 'elevation', 'landcover', 'distance_to_water', 'forest_loss', 'flooded', 'slope']


In [54]:
sampled_data = combined.sampleRegions(
    collection=sample_points,
    scale=5000,
    geometries=True
)

In [55]:
df_gee = geemap.ee_to_df(sampled_data, remove_geom=False)
print(df_gee.shape)
print(df_gee.head())

(6264, 8)
                                                 geo  distance_to_water  \
0  {'type': 'Point', 'coordinates': [8.1971269675...         161.554944   
1  {'type': 'Point', 'coordinates': [3.9301293680...         216.333077   
2  {'type': 'Point', 'coordinates': [5.2326865299...         127.279221   
3  {'type': 'Point', 'coordinates': [6.1759175783...         169.705627   
4  {'type': 'Point', 'coordinates': [5.1428550015...         624.259561   

   elevation  flooded  forest_loss  landcover      rainfall     slope  
0        730        0            1         20  13367.705167  0.240920  
1        339        0            0         40  12014.300833  0.145108  
2         14        0            1         10  23738.312557  0.010277  
3        197        0            1         20  11872.757912  0.782034  
4        295        0            1         10  17677.749280  0.999649  


In [56]:
df_gee = df_gee.drop_duplicates(subset=["rainfall", "elevation", "landcover", "distance_to_water", "forest_loss", "slope", "flooded"])
print(df_gee.shape)

(5660, 8)


In [57]:
df_gee = df_gee[df_gee["landcover"] != 80]
print(df_gee.shape)
print(df_gee["flooded"].value_counts())

(5610, 8)
flooded
0    5418
1     192
Name: count, dtype: int64


In [59]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

X = df_gee.drop(columns=["flooded", "geo"])
y = df_gee["flooded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

model_gb = GradientBoostingClassifier()
model_gb.fit(X_train, y_train)

flood_probabilities_gb = model_gb.predict_proba(X_test)[:, 1]

In [60]:
for threshold in [0.01, 0.012, 0.014, 0.016, 0.02]:
    new_predictions = (flood_probabilities_gb > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.01 ---
[[1157  198]
 [   3   45]]
              precision    recall  f1-score   support

           0       1.00      0.85      0.92      1355
           1       0.19      0.94      0.31        48

    accuracy                           0.86      1403
   macro avg       0.59      0.90      0.61      1403
weighted avg       0.97      0.86      0.90      1403


--- Threshold: 0.012 ---
[[1177  178]
 [   3   45]]
              precision    recall  f1-score   support

           0       1.00      0.87      0.93      1355
           1       0.20      0.94      0.33        48

    accuracy                           0.87      1403
   macro avg       0.60      0.90      0.63      1403
weighted avg       0.97      0.87      0.91      1403


--- Threshold: 0.014 ---
[[1193  162]
 [   5   43]]
              precision    recall  f1-score   support

           0       1.00      0.88      0.93      1355
           1       0.21      0.90      0.34        48

    accuracy             

In [61]:
for threshold in [0.012, 0.014, 0.016, 0.018, 0.02]:
    new_predictions = (flood_probabilities_gb > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.012 ---
[[1177  178]
 [   3   45]]
              precision    recall  f1-score   support

           0       1.00      0.87      0.93      1355
           1       0.20      0.94      0.33        48

    accuracy                           0.87      1403
   macro avg       0.60      0.90      0.63      1403
weighted avg       0.97      0.87      0.91      1403


--- Threshold: 0.014 ---
[[1193  162]
 [   5   43]]
              precision    recall  f1-score   support

           0       1.00      0.88      0.93      1355
           1       0.21      0.90      0.34        48

    accuracy                           0.88      1403
   macro avg       0.60      0.89      0.64      1403
weighted avg       0.97      0.88      0.91      1403


--- Threshold: 0.016 ---
[[1207  148]
 [   5   43]]
              precision    recall  f1-score   support

           0       1.00      0.89      0.94      1355
           1       0.23      0.90      0.36        48

    accuracy            

In [62]:
print(X_train.columns.tolist())


['distance_to_water', 'elevation', 'forest_loss', 'landcover', 'rainfall', 'slope']


In [63]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report
for threshold in [0.005, 0.008, 0.022, 0.025, 0.03]:
    new_predictions = (flood_probabilities_gb > threshold).astype(int)
    print(f"--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, new_predictions))
    print(classification_report(y_test, new_predictions))
    print()

--- Threshold: 0.005 ---
[[983 372]
 [  1  47]]
              precision    recall  f1-score   support

           0       1.00      0.73      0.84      1355
           1       0.11      0.98      0.20        48

    accuracy                           0.73      1403
   macro avg       0.56      0.85      0.52      1403
weighted avg       0.97      0.73      0.82      1403


--- Threshold: 0.008 ---
[[1105  250]
 [   3   45]]
              precision    recall  f1-score   support

           0       1.00      0.82      0.90      1355
           1       0.15      0.94      0.26        48

    accuracy                           0.82      1403
   macro avg       0.57      0.88      0.58      1403
weighted avg       0.97      0.82      0.88      1403


--- Threshold: 0.022 ---
[[1236  119]
 [   6   42]]
              precision    recall  f1-score   support

           0       1.00      0.91      0.95      1355
           1       0.26      0.88      0.40        48

    accuracy                

In [64]:
df_gee = df_gee.drop(columns=["slope"])
print(df_gee.columns.tolist())

['geo', 'distance_to_water', 'elevation', 'flooded', 'forest_loss', 'landcover', 'rainfall']


In [65]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

X = df_gee.drop(columns=["flooded", "geo"])
y = df_gee["flooded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

model_gb = GradientBoostingClassifier()
model_gb.fit(X_train, y_train)

flood_probabilities_gb = model_gb.predict_proba(X_test)[:, 1]

In [66]:
print(X_train.columns.tolist())

['distance_to_water', 'elevation', 'forest_loss', 'landcover', 'rainfall']


In [67]:
from sklearn.metrics import confusion_matrix, classification_report
GB_THRESHOLD = 0.016
final_predictions = (flood_probabilities_gb > GB_THRESHOLD).astype(int)

print(confusion_matrix(y_test, final_predictions))
print(classification_report(y_test, final_predictions))

[[1213  142]
 [   5   43]]
              precision    recall  f1-score   support

           0       1.00      0.90      0.94      1355
           1       0.23      0.90      0.37        48

    accuracy                           0.90      1403
   macro avg       0.61      0.90      0.66      1403
weighted avg       0.97      0.90      0.92      1403



In [68]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report

model_gb = GradientBoostingClassifier(random_state=42)
model_gb.fit(X_train, y_train)

flood_probabilities_gb = model_gb.predict_proba(X_test)[:, 1]

GB_THRESHOLD = 0.016
final_predictions = (flood_probabilities_gb > GB_THRESHOLD).astype(int)

print(confusion_matrix(y_test, final_predictions))
print(classification_report(y_test, final_predictions))

[[1214  141]
 [   5   43]]
              precision    recall  f1-score   support

           0       1.00      0.90      0.94      1355
           1       0.23      0.90      0.37        48

    accuracy                           0.90      1403
   macro avg       0.61      0.90      0.66      1403
weighted avg       0.97      0.90      0.92      1403



In [72]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
print(PROJECT_ROOT)

c:\Users\User\Desktop\FLOOD-RISK-PREDICTOR


In [73]:
GEE_DATA_PATH = PROJECT_ROOT / "data" / "gee" / "flood_features_nigeria.csv"
df_gee.to_csv(GEE_DATA_PATH, index=False)

In [74]:
import joblib

GB_MODEL_PATH = PROJECT_ROOT / "models" / "flood_classifier_nigeria_gb.pkl"

model_package = {
    "model": model_gb,
    "threshold": GB_THRESHOLD,
    "features": list(X.columns)
}

joblib.dump(model_package, GB_MODEL_PATH)

df_gee.to_csv(GEE_DATA_PATH, index=False)